In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
dataset_path = '/content/drive/MyDrive/e88186124ec611f1/dataset'

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

In [4]:
files = os.listdir(dataset_path)
print(files[:])

['sample_submission.csv', 'train.csv', 'test.csv']


In [5]:
!pip install pygeohash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.0 MB/s eta 0:00:00


In [6]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [7]:
train_df = pd.read_csv(dataset_path + '/train.csv')
test_df = pd.read_csv(dataset_path + '/test.csv')
sample_submission_df = pd.read_csv(dataset_path + '/sample_submission.csv')

In [8]:
sample_submission_df.head()

,Index,demand
0,0,0.090768
1,1,0.089885
2,2,0.007037
3,3,0.079087
4,4,0.054636


In [9]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 11.5 MB/s eta 0:00:00


In [11]:
"""
stack_model_v8.py
=================
Stacking Ensemble V8 (The Ultimate Feature-Aware Stack):
Uses the expanded 34-feature baseline from V7.
Replaces Ridge meta-learner with a Feature-Aware LightGBM meta-learner (from V5).
"""

import warnings, os, time
warnings.filterwarnings("ignore")
os.environ["PYTHONIOENCODING"] = "utf-8"

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
import pickle
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder
import pygeohash as pgh
from pygeohash import get_adjacent

USE_GPU = os.environ.get("GRIDLOCK_USE_GPU", "1") != "0"
_GPU_FALLBACK_WARNED = False


def add_lgb_device_params(params):
    params = dict(params)
    if USE_GPU:
        params["device_type"] = "gpu"
    return params


def add_xgb_device_params(params):
    params = dict(params)
    if USE_GPU:
        params["tree_method"] = "gpu_hist"
        params["predictor"] = "gpu_predictor"
    return params


def add_cb_device_params(params):
    params = dict(params)
    if USE_GPU:
        params["task_type"] = "GPU"
        params["devices"] = "0"
    return params


def fit_with_gpu_fallback(model_cls, gpu_params, cpu_params, fit_args, fit_kwargs, model_name):
    global _GPU_FALLBACK_WARNED
    if USE_GPU:
        try:
            model = model_cls(**gpu_params)
            model.fit(*fit_args, **fit_kwargs)
            return model
        except Exception as exc:
            if not _GPU_FALLBACK_WARNED:
                print(f"  [GPU fallback] {model_name} -> CPU ({exc.__class__.__name__})")
                _GPU_FALLBACK_WARNED = True

    model = model_cls(**cpu_params)
    model.fit(*fit_args, **fit_kwargs)
    return model

os.makedirs("output", exist_ok=True)
os.makedirs("output/models", exist_ok=True)   # ← folder for all pkl files

t0 = time.time()
print("=" * 70)
print("  TRAFFIC DEMAND PREDICTION -- Stacking Ensemble V8")
print("  (Expanded V7 Features + Feature-Aware Meta-Learner)")
print(f"  GPU training        : {'enabled' if USE_GPU else 'disabled'}")
print("=" * 70)

# ======================================================================
# 1. LOAD DATA
# ======================================================================
print("\n[1/7] Loading data ...")
train = train_df
test  = test_df

# ======================================================================
# 2. FEATURE ENGINEERING (V1 + V5 Features)
# ======================================================================
print("\n[2/7] Feature engineering ...")
def feature_engineer(df):
    df = df.copy()

    # Time
    hour_part = df["timestamp"].str.split(":").str[0].astype(int)
    minute_part = df["timestamp"].str.split(":").str[1].astype(int)
    df["minute"]    = hour_part * 60 + minute_part
    df["minute_sin"] = np.sin(2 * np.pi * df["minute"] / 1440)
    df["minute_cos"] = np.cos(2 * np.pi * df["minute"] / 1440)
    df["time_slot"]  = df["minute"] // 15

    df["is_peak_am"] = df["minute"].between(10 * 60, 13 * 60 + 59).astype(int)
    df["is_night"]   = df["minute"].between(0, 5 * 60 + 59).astype(int)
    df["is_evening"] = df["minute"].between(17 * 60, 21 * 60 + 59).astype(int)

    # Spatial
    gh_cache = {}
    for gh in df["geohash"].unique():
        try:
            la, lo = pgh.decode(gh)
            gh_cache[gh] = (float(la), float(lo))
        except Exception:
            gh_cache[gh] = (np.nan, np.nan)
    df["lat"] = df["geohash"].map(lambda g: gh_cache[g][0])
    df["lon"] = df["geohash"].map(lambda g: gh_cache[g][1])

    df["geo_l4"] = df["geohash"].str[:4]
    df["geo_l5"] = df["geohash"].str[:5]

    # Categorical/Ordinal
    df["LargeVehicles_bin"] = (df["LargeVehicles"] == "Allowed").astype(int)
    df["Landmarks_bin"]     = (df["Landmarks"] == "Yes").astype(int)

    road_order = {"Residential": 0, "Street": 1, "Highway": 2}
    df["RoadType_ord"] = df["RoadType"].map(road_order).fillna(-1).astype(int)
    df["IsHighVolumeLane"] = (df["NumberofLanes"] >= 4).astype(int)
    df["lane_x_road"] = df["NumberofLanes"] * (df["RoadType_ord"] + 1)

    df["road_lanes_key"] = df["RoadType_ord"].astype(str) + "_" + df["NumberofLanes"].astype(str)

    # Weather One-Hots
    weather_cats = ["Sunny", "Rainy", "Foggy", "Snowy"]
    for w in weather_cats:
        df[f"weather_{w}"] = (df["Weather"] == w).astype(int)

    # Missing Indicators
    df["RoadType_missing"] = df["RoadType"].isna().astype(int)
    df["Temp_missing"]     = df["Temperature"].isna().astype(int)
    df["Weather_missing"]  = df["Weather"].isna().astype(int)

    return df

train = feature_engineer(train)
test  = feature_engineer(test)

# ======================================================================
# 3. IMPUTATION
# ======================================================================
print("\n[3/7] Imputing missing values ...")
# Grouped median for temperature
grouped_medians = train.groupby(["geo_l4", "day"])["Temperature"].median().reset_index()
grouped_medians.columns = ["geo_l4", "day", "median_temp"]
global_temp_median = train["Temperature"].median()

def impute_temperature(df):
    df = df.copy()
    df = df.merge(grouped_medians, on=["geo_l4", "day"], how="left")
    mask = df["Temperature"].isna()
    df.loc[mask, "Temperature"] = df.loc[mask, "median_temp"]
    df["Temperature"] = df["Temperature"].fillna(global_temp_median)
    df = df.drop(columns=["median_temp"])
    return df

train = impute_temperature(train)
test  = impute_temperature(test)
train["RoadType_ord"] = train["RoadType_ord"].replace(-1, 0)
test["RoadType_ord"]  = test["RoadType_ord"].replace(-1, 0)

# ======================================================================
# 4. LAG 1D DEMAND
# ======================================================================
print("\n[4/7] Building lag_1d_demand & target encodings ...")
day48 = train[train["day"] == 48][["geohash", "time_slot", "demand"]].copy()
day48 = day48.rename(columns={"demand": "lag_1d_demand"})
day48 = day48.groupby(["geohash", "time_slot"])["lag_1d_demand"].mean().reset_index()

train = train.merge(day48, on=["geohash", "time_slot"], how="left")
train.loc[train["day"] == 48, "lag_1d_demand"] = np.nan
test = test.merge(day48, on=["geohash", "time_slot"], how="left")

# ======================================================================
# 5. TARGET ENCODING (CV-Safe)
# ======================================================================
GLOBAL_MEAN = train["demand"].mean()
N_FOLDS_TE  = 5

def smooth_target_encode(train_df, test_df, col, n_folds=5, smoothing=20):
    oof_enc  = np.zeros(len(train_df))
    test_enc = np.zeros(len(test_df))
    kf = KFold(n_folds, shuffle=True, random_state=42)
    for _, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        tr_fold = train_df.iloc[tr_idx]
        stats   = tr_fold.groupby(col)["demand"].agg(["sum", "count"])
        enc_map = (stats["sum"] + GLOBAL_MEAN * smoothing) / (stats["count"] + smoothing)
        oof_enc[val_idx] = train_df.iloc[val_idx][col].map(enc_map).fillna(GLOBAL_MEAN).values
        test_enc += test_df[col].map(enc_map).fillna(GLOBAL_MEAN).values / n_folds
    return oof_enc, test_enc

for col in ["geohash", "geo_l4", "geo_l5", "road_lanes_key"]:
    sm = 15 if col == "road_lanes_key" else 20
    tr_enc, te_enc = smooth_target_encode(train, test, col, n_folds=N_FOLDS_TE, smoothing=sm)
    train[f"{col}_te"] = tr_enc
    test[f"{col}_te"]  = te_enc

# Geohash demand variance
def smooth_target_std_encode(train_df, test_df, col, n_folds=5, smoothing=10, global_std=None):
    if global_std is None: global_std = train_df["demand"].std()
    oof_enc  = np.zeros(len(train_df))
    test_enc = np.zeros(len(test_df))
    kf = KFold(n_folds, shuffle=True, random_state=42)
    for _, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        tr_fold = train_df.iloc[tr_idx]
        stats   = tr_fold.groupby(col)["demand"].agg(["std", "count"])
        stats["std"] = stats["std"].fillna(global_std)
        enc_map = (stats["count"] * stats["std"] + smoothing * global_std) / (stats["count"] + smoothing)
        oof_enc[val_idx] = train_df.iloc[val_idx][col].map(enc_map).fillna(global_std).values
        test_enc += test_df[col].map(enc_map).fillna(global_std).values / n_folds
    return oof_enc, test_enc

tr_ghvar, te_ghvar = smooth_target_std_encode(train, test, "geohash", n_folds=N_FOLDS_TE, smoothing=10)
train["geohash_var"] = tr_ghvar
test["geohash_var"]  = te_ghvar

# Neighbor target encoding
def build_neighbor_map(geohashes):
    neighbor_map = {}
    for gh in geohashes:
        try:
            nbs = [
                get_adjacent(gh, "top"), get_adjacent(gh, "bottom"),
                get_adjacent(gh, "left"), get_adjacent(gh, "right"),
                get_adjacent(get_adjacent(gh, "left"), "top"),
                get_adjacent(get_adjacent(gh, "right"), "top"),
                get_adjacent(get_adjacent(gh, "left"), "bottom"),
                get_adjacent(get_adjacent(gh, "right"), "bottom")
            ]
            neighbor_map[gh] = nbs
        except:
            neighbor_map[gh] = []
    return neighbor_map

all_geohashes = set(train["geohash"].tolist()) | set(test["geohash"].tolist())
nb_map = build_neighbor_map(all_geohashes)

def compute_neighbor_te(train_df, test_df, nb_map, n_folds=5, smoothing=10):
    global_mean = train_df["demand"].mean()
    train_gh = train_df["geohash"].values
    test_gh  = test_df["geohash"].values
    oof_enc  = np.zeros(len(train_df))
    test_enc = np.zeros(len(test_df))
    kf = KFold(n_folds, shuffle=True, random_state=42)
    for _, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
        tr_fold = train_df.iloc[tr_idx]
        stats   = tr_fold.groupby("geohash")["demand"].agg(["sum", "count"])
        gh_map  = ((stats["sum"] + global_mean * smoothing) / (stats["count"] + smoothing)).to_dict()
        def neighbor_mean(gh):
            nbs = nb_map.get(gh, [])
            vals = [gh_map.get(n, global_mean) for n in nbs]
            return float(np.mean(vals)) if vals else global_mean
        oof_enc[val_idx]  = np.array([neighbor_mean(g) for g in train_gh[val_idx]])
        test_enc         += np.array([neighbor_mean(g) for g in test_gh]) / n_folds
    return oof_enc, test_enc

tr_nb, te_nb = compute_neighbor_te(train, test, nb_map)
train["neighbor_te"] = tr_nb
test["neighbor_te"]  = te_nb

# ======================================================================
# OHE ENCODER — fit on categorical columns, save immediately
# ======================================================================
OHE_COLS = ["RoadType", "Weather", "LargeVehicles", "Landmarks"]
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(train[OHE_COLS].fillna("Missing"))
with open("output/models/ohe_encoder.pkl", "wb") as f:
    pickle.dump(ohe, f)
print("  Saved : output/models/ohe_encoder.pkl")

FEATURE_COLS = [
    # Spatial
    "lat", "lon", "geohash_te", "geo_l4_te", "geo_l5_te", "neighbor_te",
    # Road properties
    "RoadType_ord", "NumberofLanes", "IsHighVolumeLane", "lane_x_road",
    "LargeVehicles_bin", "Landmarks_bin", "road_lanes_key_te",
    # Time properties
    "minute", "time_slot",
    "minute_sin", "minute_cos",
    "is_peak_am", "is_night", "is_evening",
    # Environment
    "Temperature", "weather_Sunny", "weather_Rainy", "weather_Foggy", "weather_Snowy",
    # Missing indicators
    "RoadType_missing", "Temp_missing", "Weather_missing",
    # Advanced targets
    "geohash_var", "lag_1d_demand"
]

X_train = train[FEATURE_COLS].values
y_train = train["demand"].values
X_test  = test[FEATURE_COLS].values

print(f"  Feature matrix size: X_train {X_train.shape} | X_test {X_test.shape}")

# ======================================================================
# 6. TUNING MODELS (15 trials each)
# ======================================================================
print("\n[5/7] Optuna Tuning (15 trials each) ...")

def tune_lgb(trial):
    params = {
        'objective': 'regression', 'metric': 'rmse',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
        'verbose': -1, 'n_jobs': -1, 'random_state': 42
    }
    gpu_params = add_lgb_device_params(params)
    kf = KFold(3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in kf.split(X_train):
        model = fit_with_gpu_fallback(
            lgb.LGBMRegressor,
            {**gpu_params, 'n_estimators': 400},
            {**params, 'n_estimators': 400},
            (X_train[tr_idx], y_train[tr_idx]),
            {'eval_set': [(X_train[val_idx], y_train[val_idx])], 'callbacks': [lgb.early_stopping(50, verbose=False)]},
            'LightGBM tuning'
        )
        preds = model.predict(X_train[val_idx])
        scores.append(r2_score(y_train[val_idx], preds))
    return np.mean(scores)

print("  Tuning LightGBM ...", end="", flush=True)
study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(tune_lgb, n_trials=15)
lgb_best = study_lgb.best_params
lgb_best.update({'objective': 'regression', 'metric': 'rmse', 'verbose': -1, 'n_jobs': -1, 'random_state': 42})
print(f" Best R2: {study_lgb.best_value:.5f}")

def tune_xgb(trial):
    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'rmse',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'n_jobs': -1, 'random_state': 42
    }
    gpu_params = add_xgb_device_params(params)
    kf = KFold(3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in kf.split(X_train):
        model = fit_with_gpu_fallback(
            xgb.XGBRegressor,
            {**gpu_params, 'n_estimators': 400},
            {**params, 'n_estimators': 400},
            (X_train[tr_idx], y_train[tr_idx]),
            {'eval_set': [(X_train[val_idx], y_train[val_idx])], 'verbose': False},
            'XGBoost tuning'
        )
        preds = model.predict(X_train[val_idx])
        scores.append(r2_score(y_train[val_idx], preds))
    return np.mean(scores)

print("  Tuning XGBoost ...", end="", flush=True)
study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(tune_xgb, n_trials=15)
xgb_best = study_xgb.best_params
xgb_best.update({'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'n_jobs': -1, 'random_state': 42})
print(f" Best R2: {study_xgb.best_value:.5f}")

def tune_cb(trial):
    params = {
        'loss_function': 'RMSE',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 5, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'verbose': False, 'random_seed': 42
    }
    gpu_params = add_cb_device_params(params)
    kf = KFold(3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in kf.split(X_train):
        model = fit_with_gpu_fallback(
            cb.CatBoostRegressor,
            {**gpu_params, 'iterations': 400},
            {**params, 'iterations': 400},
            (X_train[tr_idx], y_train[tr_idx]),
            {'eval_set': (X_train[val_idx], y_train[val_idx]), 'verbose': False},
            'CatBoost tuning'
        )
        preds = model.predict(X_train[val_idx])
        scores.append(r2_score(y_train[val_idx], preds))
    return np.mean(scores)

print("  Tuning CatBoost ...", end="", flush=True)
study_cb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_cb.optimize(tune_cb, n_trials=15)
cb_best = study_cb.best_params
cb_best.update({'loss_function': 'RMSE', 'verbose': False, 'random_seed': 42})
print(f" Best R2: {study_cb.best_value:.5f}")

# ======================================================================
# 7. BASE MODELS CV TRAINING  (+  per-fold pickle saving)
# ======================================================================
print("\n[6/7] Training Base Models (5-Fold CV) ...")
N_FOLDS = 5
kf = KFold(N_FOLDS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X_train))
oof_xgb = np.zeros(len(X_train))
oof_cb  = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_cb  = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    print(f"  Fold {fold}/{N_FOLDS} ...", end=" ")
    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[val_idx], y_train[val_idx]

    # ── LightGBM ──────────────────────────────────────────────────────
    model_lgb = fit_with_gpu_fallback(
        lgb.LGBMRegressor,
        {**add_lgb_device_params(lgb_best), 'n_estimators': 2500},
        {**lgb_best, 'n_estimators': 2500},
        (X_tr, y_tr),
        {'eval_set': [(X_va, y_va)], 'callbacks': [lgb.early_stopping(50, verbose=False)]},
        'LightGBM base model'
    )
    oof_lgb[val_idx] = model_lgb.predict(X_va)
    test_lgb += model_lgb.predict(X_test) / N_FOLDS
    with open(f"output/models/lgb_fold_{fold}.pkl", "wb") as f:
        pickle.dump(model_lgb, f)

    # ── XGBoost ───────────────────────────────────────────────────────
    model_xgb = fit_with_gpu_fallback(
        xgb.XGBRegressor,
        {**add_xgb_device_params(xgb_best), 'n_estimators': 2500, 'early_stopping_rounds': 50},
        {**xgb_best, 'n_estimators': 2500, 'early_stopping_rounds': 50},
        (X_tr, y_tr),
        {'eval_set': [(X_va, y_va)], 'verbose': False},
        'XGBoost base model'
    )
    oof_xgb[val_idx] = model_xgb.predict(X_va)
    test_xgb += model_xgb.predict(X_test) / N_FOLDS
    with open(f"output/models/xgb_fold_{fold}.pkl", "wb") as f:
        pickle.dump(model_xgb, f)

    # ── CatBoost ──────────────────────────────────────────────────────
    model_cb = fit_with_gpu_fallback(
        cb.CatBoostRegressor,
        {**add_cb_device_params(cb_best), 'iterations': 2500, 'early_stopping_rounds': 50},
        {**cb_best, 'iterations': 2500, 'early_stopping_rounds': 50},
        (X_tr, y_tr),
        {'eval_set': (X_va, y_va), 'verbose': False},
        'CatBoost base model'
    )
    oof_cb[val_idx] = model_cb.predict(X_va)
    test_cb += model_cb.predict(X_test) / N_FOLDS
    with open(f"output/models/cb_fold_{fold}.pkl", "wb") as f:
        pickle.dump(model_cb, f)

    print(f"LGBM: {r2_score(y_va, oof_lgb[val_idx]):.4f} | XGB: {r2_score(y_va, oof_xgb[val_idx]):.4f} | CB: {r2_score(y_va, oof_cb[val_idx]):.4f}")
    print(f"    Saved : lgb_fold_{fold}.pkl | xgb_fold_{fold}.pkl | cb_fold_{fold}.pkl")

r2_lgb_tot = r2_score(y_train, oof_lgb)
r2_xgb_tot = r2_score(y_train, oof_xgb)
r2_cb_tot  = r2_score(y_train, oof_cb)
print(f"\n  OOF R2 -> LGBM: {r2_lgb_tot:.5f} | XGB: {r2_xgb_tot:.5f} | CB: {r2_cb_tot:.5f}")

# ======================================================================
# 8. ADVANCED META-LEARNER  (+  per-fold pickle saving)
# ======================================================================
print("\n[7/7] Training Feature-Aware Meta-Learner (LightGBM) ...")
X_train_meta = np.column_stack([X_train, oof_lgb, oof_xgb, oof_cb])
X_test_meta  = np.column_stack([X_test, test_lgb, test_xgb, test_cb])

def tune_meta(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'random_state': 42
    }
    gpu_params = add_lgb_device_params(params)
    kf = KFold(3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in kf.split(X_train_meta):
        model = fit_with_gpu_fallback(
            lgb.LGBMRegressor,
            {**gpu_params, 'n_estimators': 300},
            {**params, 'n_estimators': 300},
            (X_train_meta[tr_idx], y_train[tr_idx]),
            {'eval_set': [(X_train_meta[val_idx], y_train[val_idx])], 'callbacks': [lgb.early_stopping(20, verbose=False)]},
            'LightGBM meta tuning'
        )
        preds = model.predict(X_train_meta[val_idx])
        scores.append(r2_score(y_train[val_idx], preds))
    return np.mean(scores)

print("  Tuning Meta LGBM ...", end="", flush=True)
study_meta = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_meta.optimize(tune_meta, n_trials=15)
meta_params = study_meta.best_params
meta_params.update({'objective': 'regression', 'metric': 'rmse', 'verbosity': -1, 'random_state': 42})
print(f" Best Meta R2: {study_meta.best_value:.5f}")

oof_stack  = np.zeros(len(X_train_meta))
test_stack = np.zeros(len(X_test_meta))
kf = KFold(5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_meta), 1):
    model_meta = fit_with_gpu_fallback(
        lgb.LGBMRegressor,
        {**add_lgb_device_params(meta_params), 'n_estimators': 1000},
        {**meta_params, 'n_estimators': 1000},
        (X_train_meta[tr_idx], y_train[tr_idx]),
        {'eval_set': [(X_train_meta[val_idx], y_train[val_idx])], 'callbacks': [lgb.early_stopping(50, verbose=False)]},
        'LightGBM meta model'
    )
    oof_stack[val_idx] = model_meta.predict(X_train_meta[val_idx])
    test_stack += model_meta.predict(X_test_meta) / 5
    with open(f"output/models/meta_lgb_fold_{fold}.pkl", "wb") as f:
        pickle.dump(model_meta, f)
    print(f"    Saved : meta_lgb_fold_{fold}.pkl")

r2_stack = r2_score(y_train, oof_stack)
print(f"  Advanced Stacking OOF R2 : {r2_stack:.6f}")

sub_path = "output/stacking_v8_submission.csv"
pd.DataFrame({
    "Index" : test["Index"].values,
    "demand": np.clip(test_stack, 0, 1),
}).to_csv(sub_path, index=False)

elapsed = time.time() - t0
print("\n" + "=" * 70)
print("  STACKING V8 PIPELINE COMPLETE")
print("=" * 70)
print(f"  Best Base Model R2 : {max(r2_lgb_tot, r2_xgb_tot, r2_cb_tot):.6f}")
print(f"  Stacking Model R2  : {r2_stack:.6f}")
print(f"  Competition Score  : {max(0, 100*r2_stack):.4f} / 100")
print(f"  Elapsed time       : {elapsed:.1f}s")
print(f"  Saved              : {sub_path}")
print("=" * 70)

# ── Final summary of all saved model files ────────────────────────────
print("\n  Model files saved to output/models/:")
saved = sorted(os.listdir("output/models"))
for fn in saved:
    print(f"    {fn}")

  TRAFFIC DEMAND PREDICTION -- Stacking Ensemble V8
  (Expanded V7 Features + Feature-Aware Meta-Learner)
  GPU training        : enabled

[1/7] Loading data ...

[2/7] Feature engineering ...

[3/7] Imputing missing values ...

[4/7] Building lag_1d_demand & target encodings ...
  Saved : output/models/ohe_encoder.pkl
  Feature matrix size: X_train (77299, 30) | X_test (41778, 30)

[5/7] Optuna Tuning (15 trials each) ...
  Tuning LightGBM ... Best R2: 0.94448
  Tuning XGBoost ...  [GPU fallback] XGBoost tuning -> CPU (XGBoostError)
 Best R2: 0.95200
  Tuning CatBoost ... Best R2: 0.94795

[6/7] Training Base Models (5-Fold CV) ...
  Fold 1/5 ... LGBM: 0.9508 | XGB: 0.9569 | CB: 0.9552
    Saved : lgb_fold_1.pkl | xgb_fold_1.pkl | cb_fold_1.pkl
  Fold 2/5 ... LGBM: 0.9534 | XGB: 0.9576 | CB: 0.9559
    Saved : lgb_fold_2.pkl | xgb_fold_2.pkl | cb_fold_2.pkl
  Fold 3/5 ... LGBM: 0.9543 | XGB: 0.9583 | CB: 0.9550
    Saved : lgb_fold_3.pkl | xgb_fold_3.pkl | cb_fold_3.pkl
  Fold 4/5 ...